# 01. RAG 개념 이해

## RAG란?

**RAG(Retrieval-Augmented Generation)** 는 LLM의 한계를 극복하기 위해  
**외부 문서를 검색(Retrieval)** 하고, 그 결과를 바탕으로 **답변을 생성(Generation)** 하는 기술임

---

## LLM의 한계

| 문제 | 설명 |
|---|---|
| 지식 단절 | 학습 데이터 이후의 최신 정보를 모름 |
| 할루시네이션 | 사실과 다른 내용을 그럴듯하게 생성 |
| 출처 불명 | 어디서 얻은 정보인지 알 수 없음 |
| 컨텍스트 제한 | 모든 문서를 프롬프트에 넣을 수 없음 |

---

## RAG 파이프라인

```
【 인덱싱 단계 (오프라인) 】

  문서 로딩  →  텍스트 분할  →  임베딩  →  벡터 스토어 저장

【 검색·생성 단계 (온라인) 】

  사용자 질문  →  임베딩  →  유사 문서 검색  →  LLM 답변 생성
```

---



## Naive RAG 구성 요소

| 단계 | 역할 | 도구 |
|---|---|---|
| Document Loader | 문서 불러오기 | TextLoader, PyPDFLoader |
| Text Splitter | 문서를 청크로 분할 | RecursiveCharacterTextSplitter |
| Embedding | 텍스트 → 벡터 변환 | OpenAIEmbeddings |
| Vector Store | 벡터 저장 및 검색 | FAISS, Chroma |
| Retriever | 관련 문서 검색 | as_retriever() |
| LLM | 최종 답변 생성 | ChatOpenAI |

## RAG vs 일반 LLM 비교 실습

In [1]:
import sys
sys.path.append("..")

from common_config import llm_connect
model = "gpt-5.4-mini"

temperatuyre = 0
max_tokens = 2086

llm = llm_connect(model = model)

In [2]:
# LLM 단독 호출 - 모델이 알고 있는 내용만 답변
question = "LangGraph는 무엇이고 어떤 개념을 사용하나요?"

response = llm.invoke(question)
print("[LLM 단독 답변]")
print(response.content)

[LLM 단독 답변]
LangGraph는 **LLM 애플리케이션을 그래프(Graph) 형태로 설계·실행하게 해주는 프레임워크**입니다.  
LangChain 위에서 동작하는 경우가 많고, 특히 **복잡한 에이전트 워크플로우**, **상태를 가지는 대화**, **반복 루프**, **분기 처리**, **사람 개입(human-in-the-loop)** 같은 패턴을 만들 때 유용합니다.

## 한 줄로 요약
- **LangChain**이 “LLM을 쓰는 도구 모음”에 가깝다면,
- **LangGraph**는 “LLM 워크플로우를 상태 기반 그래프로 설계하는 도구”에 가깝습니다.

---

## LangGraph의 핵심 개념

### 1) Graph(그래프)
애플리케이션을 **노드(node)** 와 **엣지(edge)** 로 표현합니다.

- **노드**: 실행 단위
  - 예: LLM 호출, 툴 실행, 검색, 요약, 검증
- **엣지**: 다음에 어떤 노드로 갈지 결정하는 연결
  - 예: “검색 결과가 부족하면 다시 검색”, “성공하면 종료”

즉, 단순한 선형 체인이 아니라 **조건 분기와 반복이 가능한 흐름**을 만듭니다.

---

### 2) State(상태)
LangGraph에서는 각 실행이 공유하는 **상태(state)** 를 매우 중요하게 봅니다.

예를 들어 상태에는 다음이 들어갈 수 있습니다:
- 대화 메시지 목록
- 중간 추론 결과
- 검색 결과
- 사용자 승인 여부
- 현재 단계(step)

노드들은 이 상태를 읽고, 수정하고, 다음 노드로 넘깁니다.

---

### 3) Node(노드)
노드는 보통 하나의 함수나 작업입니다.  
입력 상태를 받아서 결과를 반환합니다.

예:
- 사용자 질문 분류
- 웹 검색
- 문서 요약
- 답변 생성
- 결과 검증

---

### 4) Edge(엣지)
엣지는 노드 간 이동 경로입니다.

- **단순 엣지**: A → B
- **조건부 엣지**: 상태에 따라 A → B 또는 A → C
-


In [ ]:
# 외부 문서를 직접 제공하면 더 정확한 답변 가능
context = """
LangGraph는 LangChain의 확장으로, 그래프 기반의 멀티 에이전트 워크플로우를 구성할 수 있음
State, Node, Edge 개념을 사용해 복잡한 AI 파이프라인을 구현함
"""

# 증강
prompt = f"다음 문서를 참고해서 질문에 답하세요.\n\n문서: {context}\n\n질문: {question}"

response = llm.invoke(prompt)
print("[문서 기반 답변]")
print(response.content)

[문서 기반 답변]
LangGraph는 **LangChain의 확장 도구**로, **그래프 기반의 멀티 에이전트 워크플로우**를 구성할 수 있게 해줍니다.

사용하는 주요 개념은 다음과 같습니다:
- **State**: 작업의 상태를 저장
- **Node**: 각 처리 단계나 에이전트
- **Edge**: 노드 간의 연결과 흐름

즉, LangGraph는 이 개념들을 이용해 **복잡한 AI 파이프라인**을 만들 수 있습니다.


---

## 정리

- RAG는 **외부 문서 → 검색 → 생성** 3단계로 동작함
- 다음 노트북부터 각 단계를 하나씩 실습함

    ```
    02_document_loader   → 문서 불러오기
    03_text_splitter     → 텍스트 분할
    04_embedding         → 임베딩
    05_vector_store      → 벡터 스토어
    06_retriever         → 검색
    07_rag_pipeline      → 전체 파이프라인
    ```